## Name: Blessing Adeniji
##  Degree: MSc Artifical Intelligence Online
## Capstone Project: AI-Generated Text Detection - Deepfakes
Final Step: Error Analysis - Study the texts the detectors get wrong.

This notebook takes the trained models, run them over test sets, and keeps them per-sample predictions, 
then compare lingusitic features (sentence length, vocabulary diversity, punctuation) of wrong vs right predictions.

Analysis groups:
- Encoder vs decoder on the same hard task (MAGE-trained, tested on RAID) - do the two architectures fail on the SAME texts?
- The worst collapse (Abstracts-trained model on MAGE, 51% acc) - what does total generalisation failure look like linguistically?
- The best model's remaining errors (MAGE-trained on MAGE, ~5% error) - what stays hard even in the best case?




In [1]:
# Step 1: Get per-sample predictions.
# trainer.evaluate() only returns overall scores, so this uses
# trainer.predict() instead, which returns the prediction for EVERY text.
# Saved per row: the text, the true label, and each model's predicted label.
# A row where prediction != true label is a misclassification -
# these rows are the raw material for the whole analysis.

import os, pandas as pd, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding


In [3]:
# Load the RAID test set
raid_test_df = pd.read_csv("data_splits/RAID_test.csv")

# The two MAGE-trained models to compare
models_to_analyse = {
    "encoder": "models/ettin68m_mage_final",
    "decoder": "models/decoder_mage_final",   
}

# Results table with text and true label
predictions_df = raid_test_df[["text", "label"]].copy()

for name, path in models_to_analyse.items():
    # Load this model and tokenizer from disk
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path)
    
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)
    test_ds = Dataset.from_pandas(raid_test_df).map(tokenize, batched=True)

    # Prediction-only trainer
    trainer = Trainer(
        model=model, 
        args=TrainingArguments(output_dir="tmp_predict", per_device_eval_batch_size=32, report_to="none"), 
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    )

    # Predict() returns raw scores for every sample; argmax picks the predicted clas
    output = trainer.predict(test_ds)
    predictions_df[f"{name}_pred"] = np.argmax(output.predictions, axis=1)
    print(f"{name}: done - accuracy {(predictions_df[f'{name}_pred'] == predictions_df['label']).mean():.4f}")

# check the correctness in each model
predictions_df["encoder_correct"] = predictions_df["encoder_pred"] == predictions_df["label"]
predictions_df["decoder_correct"] = predictions_df["decoder_pred"] == predictions_df["label"]

# Save the full per-sample table
os.makedirs("error_analysis", exist_ok=True)
predictions_df.to_csv("error_analysis/mage_models_on_raid_predictions.csv", index=False)

# The headline question: do they fail on the SAME texts?
both_enc_dec_wrong = (~predictions_df["encoder_correct"] & ~predictions_df["decoder_correct"]).sum()
only_encoder_wrong = (~predictions_df["encoder_correct"] & predictions_df["decoder_correct"]).sum()
only_decoder_wrong = (predictions_df["encoder_correct"] & ~predictions_df["decoder_correct"]).sum()
both_enc_dec_right = (predictions_df["encoder_correct"] & predictions_df["decoder_correct"]).sum()

# Print results
print(f"\nBoth right: {both_enc_dec_right}")
print(f"Both wrong: {both_enc_dec_wrong}")
print(f"Only encoder wrong: {only_encoder_wrong}")
print(f"Only decoder wrong: {only_decoder_wrong}")

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

encoder: done - accuracy 0.7715


Loading weights:   0%|          | 0/157 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

decoder: done - accuracy 0.7804

Both right: 32408
Both wrong: 8113
Only encoder wrong: 1946
Only decoder wrong: 1554


In [6]:
# Step 2: Lingusitic features of errors and correct predictions
# For each text, compute: word count, average sentence length, type-token, ratio (vocabulary diversity), and punctuation density.
# Compare the feature averages across the four outcome groups from step 1

# re = regular expression for regex - mini lanaguage for pattern matching in text.
import re

# Load the per-sample predictions from step1
df = pd.read_csv("error_analysis/mage_models_on_raid_predictions.csv")

# Feature functions
def word_count(text):
    # No. of word in text
    return len(str(text).split())

def avg_sentence_length(text):
    # Split on sentence-ending punctuation: average words per sentence
    sentences = [s for s in re.split(r"[.!?]+", str(text)) if s.strip()]
    if not sentences: 
        return 0
    return np.mean([len(s.split()) for s in sentences])

def type_token_ratio(text):
    # Unique words / total words - higer = more vared vocabulary
    words = str(text).lower().split()
    if not words:
        return 0
    return len(set(words)) / len(words)

def punctuation_density(text):
    # Punctuation marks per 100 characters
    text = str(text)
    if not text:
        return 0
    punctuation = len(re.findall(r"[.,;:!?\"'()\-]", text))
    return 100 * punctuation / len(text)

# Compute all four features for every text (44k rows)
df["word_count"] = df["text"].apply(word_count)
df["avg_sentence_length"] = df["text"].apply(avg_sentence_length)
df["type_token_ratio"] = df["text"].apply(type_token_ratio)
df["punctuation_density"] = df["text"].apply(punctuation_density)

# Assign each row to its outcome group
def outcome_group(row):
    if row["encoder_correct"] and row["decoder_correct"]:
        return "both_enc_dec_right"
    if not row["encoder_correct"] and not row["decoder_correct"]:
        return "both_enc_dec_wrong"
    if not row["encoder_correct"]:
        return "only_encoder_wrong"
    return "only_decoder_wrong"

df["group"] = df.apply(outcome_group, axis=1)

# Compare feature averages per group
summary = df.groupby("group")[["word_count", "avg_sentence_length", "type_token_ratio", "punctuation_density"]].mean().round(2)
print(summary)

# Splits by true label - are errors mostly on human or AI texts?
print("\nError counts by true label )0=human, 1=AI):")
print(df[df["group"] == "both_enc_dec_wrong"]["label"].value_counts())

# Save the table for further analysis
df.to_csv("error_analysis/raid_predictions_with_features.csv", index=False)

                    word_count  avg_sentence_length  type_token_ratio  \
group                                                                   
both_enc_dec_right      265.24                26.12              0.62   
both_enc_dec_wrong      275.70                23.32              0.61   
only_decoder_wrong      242.98                21.85              0.65   
only_encoder_wrong      237.87                23.99              0.65   

                    punctuation_density  
group                                    
both_enc_dec_right                 2.53  
both_enc_dec_wrong                 2.41  
only_decoder_wrong                 2.76  
only_encoder_wrong                 2.83  

Error counts by true label )0=human, 1=AI):
label
0    6389
1    1724
Name: count, dtype: int64


In [7]:
# Step 3: This is to sharpen step2's findings
# False-flag rate: what fraction of HUMAN texts do the models wrongly call AI?
# Fairer feature comparison: compare error vs correct texts WITHIN the same true label 
# signifance test (Mann-Whitney U) on the key feature difference
from scipy.stats import mannwhitneyu

df = pd.read_csv("error_analysis/raid_predictions_with_features.csv")

# False flag rates
human_texts = df[df["label"] == 0]
ai_texts = df[df["label"] == 1]

# Print results
print("Total human texts:", len(human_texts), "| Total AI texts:", len(ai_texts))
print(f"Encoder false-flag rate (human called AI): {(human_texts['encoder_pred'] == 1).mean():.3f}")
print(f"Decoder false-flag rate (human called AI): {(human_texts['decoder_pred'] == 1).mean():.3f}")
print(f"Encoder missed-AI rate (AI called human): {(ai_texts['encoder_pred'] == 0).mean():.3f}")
print(f"Decoder missed-AI rate (AI called human): {(ai_texts['decoder_pred'] == 0).mean():.3f}")

# Feature comparison within Human texts only
human_right = df[(df["label"] == 0) & (df["group"] == "both_enc_dec_right")]
human_wrong = df[(df["label"] == 0) & (df["group"] == "both_enc_dec_wrong")]

features = ["word_count", "avg_sentence_length", "type_token_ratio", "punctuation_density"]
print("\nHuman texts: correctly classified vs falsely flagged (means)")
comparison = pd.DataFrame({
    "correct (human)": human_right[features].mean().round(2),
    "false-flagged (human)": human_wrong[features].mean().round(2),
})
print(comparison)

# Significance tests
print("\nMann-Whitney U tests (human correct vs human false-flagged):")
for f in features:
    stat, p = mannwhitneyu(human_right[f], human_wrong[f])
    print(f"  {f}: p = {p:.2e} {'(significant)' if p < 0.01 else '(not significant)'}")



Total human texts: 22011 | Total AI texts: 22010
Encoder false-flag rate (human called AI): 0.330
Decoder false-flag rate (human called AI): 0.327
Encoder missed-AI rate (AI called human): 0.127
Decoder missed-AI rate (AI called human): 0.112

Human texts: correctly classified vs falsely flagged (means)
                     correct (human)  false-flagged (human)
word_count                    301.63                 284.68
avg_sentence_length            24.45                  22.19
type_token_ratio                0.63                   0.60
punctuation_density             2.59                   2.31

Mann-Whitney U tests (human correct vs human false-flagged):
  word_count: p = 3.12e-33 (significant)
  avg_sentence_length: p = 1.95e-200 (significant)
  type_token_ratio: p = 8.06e-131 (significant)
  punctuation_density: p = 2.44e-74 (significant)
